# Modbus — master, slave, dan simulator

**Apa itu.** Modbus adalah bahasa request/response yang umum di PLC, meter, drive, dan sensor. *Master* (client) bertanya; *slave* (server) menjawab dari empat tabel: coil, discrete input, input register, holding register.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.9.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.9.0-preview.1"

## Server: PLC virtual
`ModbusSimulator.CreateVirtualPlc` menganimasikan suhu, kecepatan motor, daya, dan penghitung produksi.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Modbus;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
var store = new ModbusDataStore();
var plcServer = ModbusServer.Create(o => o.ListenInMemory(link).WithStore(store));
var simulator = ModbusSimulator.CreateVirtualPlc(store);
await plcServer.StartAsync();
for (var i = 0; i < 20; i++) simulator.Tick(0.25); // deterministic time
Console.WriteLine($"temperature register = {store.InputRegisters[0]}");

## Client: baca dan konversi
Register berukuran 16-bit; float 32-bit memakai dua register. `ModbusConvert` menangani empat urutan word yang ditemui di lapangan.

In [ ]:
var plc = ModbusClient.Create(o => o.UseInMemory(link).WithUnitId(1));
var ir = await plc.ReadInputRegistersAsync(0, 8);
Console.WriteLine($"temperature {ir[0] / 10.0} °C, rpm {ir[3]}, power {ModbusConvert.ToSingle(ir.AsSpan(4, 2)):0.00} kW");

## Tulis — dan mode aman read-only
Penulisan mengubah peralatan sungguhan. `.AsReadOnly()` memblokirnya sebelum apa pun sampai ke jalur.

In [ ]:
await plc.WriteSingleCoilAsync(0, false); // stop the motor
for (var i = 0; i < 20; i++) simulator.Tick(0.25);
Console.WriteLine($"rpm after stop: {(await plc.ReadInputRegistersAsync(3, 1))[0]}");

var monitor = ModbusClient.Create(o => o.UseInMemory(link).AsReadOnly());
try { await monitor.WriteSingleRegisterAsync(0, 300); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }

## Mesin Rust
`NativeModbusClient` menjalankan protokol yang sama lewat state machine Rust sans-I/O. API sama, byte di jalur identik.

In [ ]:
using IoTCom.Net.Native.Modbus;
if (NativeModbusMaster.IsSupported)
{
    await using var rust = NativeModbusClient.Create(o => o.UseInMemory(link));
    Console.WriteLine($"via Rust: {string.Join(", ", await rust.ReadHoldingRegistersAsync(0, 3))}");
}
else Console.WriteLine("Native library not available for this platform — the managed client covers everything.");

## Pemecahan masalah
- **Timeout**: unit id salah, port salah, atau parity/baud RTU tidak cocok (default Modbus RTU adalah 8E1).
- **IllegalDataAddress**: alamat + jumlah melewati peta perangkat — periksa basis 0 vs basis 1 (40001 = HR0).
- **Data kacau pada RTU**: dua master di satu bus, atau resistor terminasi tidak terpasang.

## Latihan
Tambahkan `simulator.AddSignal(ModbusTable.HoldingRegisters, 50, t => Math.Sin(t) * 100 + 100)` lalu plot HR50 terhadap waktu.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*